In [ ]:
pip install ruptures

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 18.7 MB/s eta 0:00:00


In [ ]:
"""
=============================================================================
Phase 3 — Diagnostic Analysis: The War on Cash
India's Digital Payment Ecosystem  |  RBI Table-45  |  Nov-2019 → Mar-2025
=============================================================================
Produces 7 publication-ready PNGs in plots_phase3/

Plots
  P1  — Event Overlay Timeline          (clean log-scale, no text collisions)
  P2  — COVID Impact  Δ% bars           (trend-corrected, side annotations)
  P3  — Interchange Impact Δ% bars      (Zero-MDR + PPI Cap)
  P4  — Pareto Stacked + FY2025 Pareto
  P4c — Top-3 Share Trajectory
  PSTAT — Stationarity enforcement
  PBREAK — Structural break (Chow + PELT)
  PGRANGER — Granger causality heatmap  (clean grid, no bar overlap)

Statistical rigour
  [C1] Welch t-test on first-differenced series (removes autocorrelation)
  [C2] Trend-corrected Δ%: actual post vs trend-extrapolated post
  [C3] Apr-2023 pre-window capped to pre-break regime
  [C4] Interpretation: sign + significance only, no mechanism claims
  [C5] Debit / NEFT labelled as imperfect proxies
  [C6] Cohen's d reported alongside p-value
  [C7] Stability flag: sign-flip at ±12 m vs ±6 m
=============================================================================
"""

import warnings, sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.ticker as mticker
import matplotlib.dates  as mdates
import matplotlib.gridspec as gridspec
from matplotlib.lines   import Line2D
from scipy              import stats as scipy_stats
from statsmodels.tsa.stattools          import adfuller, grangercausalitytests
from statsmodels.regression.linear_model import OLS
from statsmodels.tools                  import add_constant

try:
    import ruptures as rpt
    HAS_RUPTURES = True
except ImportError:
    HAS_RUPTURES = False

warnings.filterwarnings("ignore")

# ─────────────────────────────────────────────────────────────────────────────
# CONFIG
# ─────────────────────────────────────────────────────────────────────────────
INPUT_CSV  = "rbi_payments_clean.csv"
OUT_DIR    = Path("plots_phase3")
DPI        = 180
ALPHA      = 0.05
BREAK_DATE = pd.Timestamp("2023-04-01")
DATA_START = pd.Timestamp("2019-11-01")

# ── Refined palette — works on white paper, readable when printed ──
PALETTE = {
    "UPI":    "#D62728",   # vivid red
    "NEFT":   "#1F77B4",   # steel blue
    "IMPS":   "#17A589",   # teal
    "RTGS":   "#7B2D8B",   # plum
    "Credit": "#E67E22",   # amber
    "Debit":  "#555555",   # neutral grey
    "bg":     "#FAFAFA",
    "grid":   "#E8E8E8",
    "text":   "#1A1A2E",
    "anno":   "#FFFDE7",
    "anno_border": "#C9A800",
}

ALL_MODES = ["UPI", "NEFT", "IMPS", "RTGS", "Credit", "Debit"]

EVENTS = [
    dict(label="Zero-MDR\nMandate", date="2020-01-01", color="#1A5276", side="top"),
    dict(label="COVID\nLockdown",   date="2020-03-01", color="#C0392B", side="top"),
    dict(label="Full\nLockdown",    date="2020-04-01", color="#922B21", side="bottom"),
    dict(label="PPI\nInterchange\nCap",  date="2023-04-01", color="#D35400", side="top"),
]

FY_BANDS = {
    "FY20": ("2019-04-01", "2020-03-31"),
    "FY21": ("2020-04-01", "2021-03-31"),
    "FY22": ("2021-04-01", "2022-03-31"),
    "FY23": ("2022-04-01", "2023-03-31"),
    "FY24": ("2023-04-01", "2024-03-31"),
    "FY25": ("2024-04-01", "2025-03-31"),
}

CHOW_CANDIDATES = ["2020-01-01","2020-03-01","2020-04-01","2021-01-01","2023-04-01"]

# ─────────────────────────────────────────────────────────────────────────────
# MATPLOTLIB STYLE
# ─────────────────────────────────────────────────────────────────────────────
plt.rcParams.update({
    "figure.facecolor":  PALETTE["bg"],
    "axes.facecolor":    PALETTE["bg"],
    "axes.edgecolor":    "#CCCCCC",
    "axes.labelcolor":   PALETTE["text"],
    "axes.grid":         True,
    "grid.color":        PALETTE["grid"],
    "grid.linewidth":    0.7,
    "xtick.color":       PALETTE["text"],
    "ytick.color":       PALETTE["text"],
    "text.color":        PALETTE["text"],
    "font.family":       "DejaVu Sans",
    "font.size":         9,
    "axes.titlesize":    11,
    "axes.titleweight":  "bold",
    "axes.labelsize":    9,
    "legend.framealpha": 0.92,
    "legend.edgecolor":  "#CCCCCC",
    "savefig.facecolor": "white",
})

# ─────────────────────────────────────────────────────────────────────────────
# UTILITY
# ─────────────────────────────────────────────────────────────────────────────
def savefig(fig, name):
    OUT_DIR.mkdir(exist_ok=True)
    p = OUT_DIR / name
    fig.savefig(p, dpi=DPI, bbox_inches="tight", facecolor="white")
    print(f"  ✓  {p}")
    plt.close(fig)

def sig_stars(p):
    if p < 0.001: return "***"
    if p < 0.01:  return "**"
    if p < 0.05:  return "*"
    return "ns"

def _fmt_crore(x, _):
    if x >= 1e9: return f"{x/1e9:.1f}B"
    if x >= 1e7: return f"{x/1e7:.0f}Cr"
    return f"{x:.0f}"

def _add_event_lines(ax, xlim=None):
    """
    Draw event vlines + offset labels in a non-colliding way.
    Labels placed at alternating y positions using side='top'/'bottom'.
    """
    ylo, yhi = ax.get_ylim()
    span = yhi - ylo
    offsets = {"top": 0.93, "bottom": 0.07}
    vas     = {"top": "top", "bottom": "bottom"}

    for ev in EVENTS:
        xd = pd.Timestamp(ev["date"])
        if xlim and not (xlim[0] <= xd <= xlim[1]):
            continue
        ax.axvline(xd, color=ev["color"], lw=1.4, ls=(0,(4,3)), alpha=0.85, zorder=4)
        yrel = offsets[ev["side"]]
        ax.text(
            xd, ylo + span * yrel, ev["label"],
            color=ev["color"], fontsize=6.2, rotation=90,
            va=vas[ev["side"]], ha="right",
            bbox=dict(boxstyle="round,pad=0.18", fc="white", alpha=0.75, ec="none"),
            zorder=6,
        )

def _insight_box(ax, text, x=0.02, y=0.03, fontsize=7.5):
    ax.text(
        x, y, text, transform=ax.transAxes,
        fontsize=fontsize, va="bottom", ha="left",
        color=PALETTE["text"],
        bbox=dict(boxstyle="round,pad=0.5", fc=PALETTE["anno"],
                  ec=PALETTE["anno_border"], alpha=0.95, lw=0.9),
        zorder=10, linespacing=1.55,
    )

def _source_note(fig, note="Source: RBI Table-45  |  Phase 3 Diagnostic Analysis"):
    fig.text(0.99, 0.005, note, ha="right", fontsize=6.5, color="#888888", style="italic")

# ─────────────────────────────────────────────────────────────────────────────
# STATISTICS ENGINE
# ─────────────────────────────────────────────────────────────────────────────
def _seasonal_diff(s, lag=12):
    return s.diff(lag).dropna()

def _enforce_stationarity(s, alpha=0.05, max_d=2):
    sd = _seasonal_diff(s)
    current, d = sd.copy(), 0
    while d <= max_d:
        stat, p, *_ = adfuller(current.dropna(), autolag="AIC")
        if p < alpha:
            return current, d, p
        current = current.diff().dropna()
        d += 1
    return current, d, p

def _ttest_diff(pre, post):
    pd_ = pre.diff().dropna()
    po_ = post.diff().dropna()
    if len(pd_) < 2 or len(po_) < 2:
        if len(pre) >= 2 and len(post) >= 2:
            t, p = scipy_stats.ttest_ind(pre, post, equal_var=False)
            return t, p, True
        return np.nan, 1.0, True
    t, p = scipy_stats.ttest_ind(pd_, po_, equal_var=False)
    return t, p, False

def _effect_size(pre, post):
    pd_ = pre.diff().dropna()
    po_ = post.diff().dropna()
    if len(pd_) < 2 or pd_.std() == 0:
        return np.nan
    return (po_.mean() - pd_.mean()) / pd_.std()

def _trend_corrected(pre, post):
    xp = np.arange(len(pre))
    c  = np.polyfit(xp, pre.values, 1)
    xo = np.arange(len(pre), len(pre) + len(post))
    exp_mean = np.polyval(c, xo).mean()
    act_mean = post.mean()
    tc_pct   = (act_mean - exp_mean) / abs(exp_mean) * 100 if exp_mean else np.nan
    return act_mean, exp_mean, tc_pct

def _get_window(df, event_date, window_m, col="Volume"):
    ets   = pd.Timestamp(event_date)
    pre_s = max(ets - pd.DateOffset(months=window_m), DATA_START)
    post_e = ets + pd.DateOffset(months=window_m)
    out = {}
    for mode, grp in df.groupby("Payment_Mode"):
        g    = grp.set_index("Date").sort_index()
        pre  = g.loc[(g.index >= pre_s) & (g.index < ets),   col]
        post = g.loc[(g.index >= ets)   & (g.index < post_e), col]
        if len(pre) >= 2 and len(post) >= 2:
            out[mode] = (pre, post)
    return out

def compute_event_stats(df, event_date, col="Volume"):
    w6  = _get_window(df, event_date, 6,  col)
    w12 = _get_window(df, event_date, 12, col)
    out = {}
    for mode in ALL_MODES:
        if mode not in w6:
            continue
        pre6, post6       = w6[mode]
        t6, p6, lim6      = _ttest_diff(pre6, post6)
        d6                = _effect_size(pre6, post6)
        act6, exp6, tc6   = _trend_corrected(pre6, post6)
        raw6 = (post6.mean() - pre6.mean()) / pre6.mean() * 100 if pre6.mean() else np.nan
        stable = True
        tc12   = np.nan
        if mode in w12:
            pre12, post12 = w12[mode]
            _, _, tc12    = _trend_corrected(pre12, post12)
            if not (np.isnan(tc6) or np.isnan(tc12)):
                stable = np.sign(tc6) == np.sign(tc12)
        out[mode] = dict(
            pre_mean=pre6.mean(), post_mean=post6.mean(), exp_mean=exp6,
            raw_pct=raw6, tc_pct=tc6, tc_pct12=tc12,
            t=t6, p=p6, d=d6,
            sig=bool(p6 < ALPHA) if not np.isnan(p6) else False,
            limited=lim6, stable=stable,
            n_pre=len(pre6), n_post=len(post6),
        )
    return out

# ─────────────────────────────────────────────────────────────────────────────
# DELTA-% PANEL  (clean, no collisions)
# ─────────────────────────────────────────────────────────────────────────────
def _draw_delta_panel(ax, event_stats, title, show_raw=True):
    """
    One bar per mode = trend-corrected Δ%.
    Annotations placed OUTSIDE bars to avoid collision:
      positive bar → label above bar top
      negative bar → label below bar bottom
    Raw Δ% shown as dashed outline.
    """
    modes = [m for m in ALL_MODES if m in event_stats]
    x     = np.arange(len(modes))
    width = 0.55

    tc_vals, raw_vals, colors, sig_flags = [], [], [], []
    for m in modes:
        s = event_stats[m]
        tc_vals.append(s["tc_pct"])
        raw_vals.append(s["raw_pct"])
        colors.append(PALETTE[m])
        sig_flags.append(s["sig"])

    # ── bars ──
    bars = []
    for i, (tc, col, sig) in enumerate(zip(tc_vals, colors, sig_flags)):
        if np.isnan(tc):
            bars.append(None)
            continue
        bar = ax.bar(i, tc, color=col,
                     alpha=0.85 if sig else 0.25,
                     edgecolor=col, linewidth=0.8,
                     width=width, zorder=3)
        bars.append(bar)

    # raw Δ% dashed outline
    if show_raw:
        for i, (raw, col) in enumerate(zip(raw_vals, colors)):
            if np.isnan(raw):
                continue
            ax.bar(i, raw, color="none", edgecolor=col,
                   linewidth=1.3, width=width,
                   linestyle=(0,(4,2)), zorder=4)

    ax.axhline(0, color="#333333", lw=1.0, zorder=2)

    # ── annotations OUTSIDE bars, never overlapping ──
    vals_clean = [v for v in tc_vals if not np.isnan(v)]
    data_range = max(abs(min(vals_clean)), abs(max(vals_clean))) if vals_clean else 10
    step = max(data_range * 0.08, 3.0)  # dynamic spacing

    for i, m in enumerate(modes):
        s  = event_stats[m]
        tc = s["tc_pct"]
        p  = s["p"]
        d  = s["d"]
        if np.isnan(tc):
            continue

        sig   = s["sig"]
        lim   = s["limited"]
        stb   = s["stable"]
        color = PALETTE[m]

        # Build compact annotation
        d_str   = f"d={d:.2f}" if not np.isnan(d) else ""
        flag    = ("⚑LIM " if lim else "") + ("⚑UNSTB" if not stb else "")
        stars   = sig_stars(p) if not np.isnan(p) else "—"
        line1   = f"{'+' if tc>=0 else ''}{tc:.1f}%"
        line2   = f"p={p:.3f} {stars}"
        line3   = d_str
        ann_txt = "\n".join(filter(None, [line1, line2, line3, flag.strip()]))

        # Position: outside the bar
        if tc >= 0:
            ypos = tc + step
            va   = "bottom"
        else:
            ypos = tc - step
            va   = "top"

        ax.text(i, ypos, ann_txt,
                ha="center", va=va, fontsize=6.8,
                color=color if sig else "#999999",
                fontweight="bold" if sig else "normal",
                linespacing=1.35, zorder=7)

    # ── dynamic y-limits with generous headroom ──
    if vals_clean:
        lo = min(vals_clean)
        hi = max(vals_clean)
        pad = (max(abs(lo), abs(hi)) + data_range) * 0.7 + 12
        ax.set_ylim(lo - pad, hi + pad)

    # ── proxy labels on x-axis ──
    xlabels = []
    for m in modes:
        if m == "Debit":
            xlabels.append("Debit\n(⚠ proxy)")
        elif m == "NEFT":
            xlabels.append("NEFT\n(⚠ proxy)")
        else:
            xlabels.append(m)
    ax.set_xticks(x)
    ax.set_xticklabels(xlabels, fontsize=8)

    ax.set_ylabel("Trend-corrected Δ%", fontsize=8)
    ax.set_title(title, fontsize=10, fontweight="bold", pad=10)
    ax.grid(axis="y", ls=":", alpha=0.4, zorder=1)
    ax.set_axisbelow(True)

    # legend
    handles = [
        mpatches.Patch(fc="grey", alpha=0.85, label="Sig. (α=0.05) — solid"),
        mpatches.Patch(fc="grey", alpha=0.25, label="Not sig. — faded"),
        Line2D([0],[0], color="grey", lw=1.3, ls=(0,(4,2)), label="Raw pre→post Δ%"),
    ]
    ax.legend(handles=handles, fontsize=7, loc="upper right",
              handlelength=1.8, framealpha=0.92)

    # method footnote inside axes (very small, bottom-left corner)
    ax.text(0.01, 0.01,
            "[C1] 1st-diff Welch t-test  [C2] Trend-extrapolated baseline  "
            "[C6] Cohen's d  [C7] ⚑UNSTB=sign flip ±12m",
            transform=ax.transAxes, fontsize=5.5, color="#888888",
            va="bottom", ha="left", zorder=8)

# ─────────────────────────────────────────────────────────────────────────────
# FY SHARES (shared helper)
# ─────────────────────────────────────────────────────────────────────────────
def fy_shares(df):
    rows = []
    for fy, (s, e) in FY_BANDS.items():
        sub   = df[(df["Date"] >= s) & (df["Date"] <= e)]
        total = sub["Volume"].sum()
        for mode, grp in sub.groupby("Payment_Mode"):
            rows.append(dict(FY=fy, Mode=mode,
                             Volume=grp["Volume"].sum(),
                             Value=grp["Value"].sum(),
                             Share=grp["Volume"].sum()/total*100 if total else 0))
    return pd.DataFrame(rows)

# ═════════════════════════════════════════════════════════════════════════════
#  P1 — EVENT OVERLAY TIMELINE
# ═════════════════════════════════════════════════════════════════════════════
def plot_p1(df):
    fig = plt.figure(figsize=(15, 8))
    gs  = gridspec.GridSpec(2, 1, height_ratios=[3, 1.1], hspace=0.08)
    ax1 = fig.add_subplot(gs[0])
    ax2 = fig.add_subplot(gs[1], sharex=ax1)

    fig.suptitle(
        "P1 — India Payment Volume Timeline  ·  Nov 2019 – Mar 2025\n"
        "Log scale  |  Major policy events overlaid",
        fontsize=13, fontweight="bold", y=1.01, color=PALETTE["text"],
    )

    # Main time series
    for mode in ["UPI","NEFT","IMPS","Credit","Debit"]:
        sub = df[df["Payment_Mode"]==mode].set_index("Date").sort_index()
        ax1.semilogy(sub.index, sub["Volume"]/1e7,
                     color=PALETTE[mode], lw=2.3, label=mode, zorder=3)

    ax1.axvspan("2020-03-01","2020-06-30", color="#C0392B", alpha=0.06, zorder=1)
    ax1.set_ylabel("Monthly Volume  (Crore transactions, log scale)", fontsize=9)
    ax1.yaxis.set_major_formatter(mticker.FuncFormatter(_fmt_crore))
    ax1.legend(loc="upper left", fontsize=9, ncol=3, framealpha=0.95)
    ax1.grid(True, which="both", ls=":", alpha=0.35)
    _add_event_lines(ax1)

    # Bottom strip — Debit detail
    deb = df[df["Payment_Mode"]=="Debit"].set_index("Date").sort_index()
    ax2.fill_between(deb.index, deb["Volume"]/1e7,
                     alpha=0.25, color=PALETTE["Debit"])
    ax2.plot(deb.index, deb["Volume"]/1e7,
             color=PALETTE["Debit"], lw=1.8)
    ax2.axvspan("2020-03-01","2020-06-30", color="#C0392B", alpha=0.06)
    ax2.set_ylabel("Debit  (Cr)\n⚠ proxy", fontsize=7.5)
    ax2.yaxis.set_major_formatter(mticker.FuncFormatter(_fmt_crore))
    ax2.grid(True, ls=":", alpha=0.35)
    _add_event_lines(ax2)

    for ax in (ax1, ax2):
        ax.xaxis.set_major_locator(mdates.MonthLocator(bymonth=[1,4,7,10]))
        ax.xaxis.set_major_formatter(mdates.DateFormatter("%b-%y"))
        plt.setp(ax.xaxis.get_majorticklabels(), rotation=35, ha="right", fontsize=7)

    plt.setp(ax1.xaxis.get_majorticklabels(), visible=False)

    _insight_box(ax1,
        "Key observations\n"
        "• UPI overtook NEFT by volume in FY2021\n"
        "• COVID Full Lockdown (Apr 2020): sharp volume dip across all modes\n"
        "• Post-unlock: UPI growth re-accelerated; Debit flat\n"
        "• PPI Cap (Apr 2023): visible kink in IMPS / Credit trajectories",
        x=0.55, y=0.04, fontsize=5,
    )

    _source_note(fig)
    plt.tight_layout()
    savefig(fig, "P1_event_overlay_timeline.png")

# ═════════════════════════════════════════════════════════════════════════════
#  P2 — COVID IMPACT
# ═════════════════════════════════════════════════════════════════════════════
def plot_p2(df):
    events = {
        "COVID Partial Lockdown  (Mar 2020)": "2020-03-01",
        "COVID Full Lockdown  (Apr 2020)":    "2020-04-01",
    }
    fig, axes = plt.subplots(1, 2, figsize=(16, 7.5))
    fig.patch.set_facecolor("white")
    fig.suptitle(
        "P2 — COVID-19 Event Impact on Payment Modes\n"
        "Trend-corrected Δ%  ·  ±6-month window  ·  1st-diff Welch t-test  ·  α = 0.05",
        fontsize=12, fontweight="bold", color=PALETTE["text"],
    )
    for ax, (label, date) in zip(axes, events.items()):
        stats = compute_event_stats(df, date)
        _draw_delta_panel(ax, stats, label)

    # Shared insight
    axes[0].set_xlabel("Payment Mode", fontsize=9)
    axes[1].set_xlabel("Payment Mode", fontsize=9)

    _source_note(fig)
    plt.tight_layout()
    savefig(fig, "P2_covid_impact.png")

# ═════════════════════════════════════════════════════════════════════════════
#  P3 — INTERCHANGE IMPACT
# ═════════════════════════════════════════════════════════════════════════════
def plot_p3(df):
    events = {
        "Zero-MDR Mandate  (Jan 2020)\n⚠ Limited pre-data (n≈2 months)":  "2020-01-01",
        "PPI Interchange Cap  (Apr 2023)\n[C3] Pre-window = pre-break regime only": "2023-04-01",
    }
    fig, axes = plt.subplots(1, 2, figsize=(16, 7.5))
    fig.patch.set_facecolor("white")
    fig.suptitle(
        "P3 — UPI Interchange Regulation Impact\n"
        "Trend-corrected Δ%  ·  ±6-month window  ·  α = 0.05",
        fontsize=12, fontweight="bold", color=PALETTE["text"],
    )
    for ax, (label, date) in zip(axes, events.items()):
        stats = compute_event_stats(df, date)
        _draw_delta_panel(ax, stats, label)
        ax.set_xlabel("Payment Mode", fontsize=9)

    _source_note(fig)
    plt.tight_layout()
    savefig(fig, "P3_interchange_impact.png")

# ═════════════════════════════════════════════════════════════════════════════
#  P4 — PARETO ANALYSIS
# ═════════════════════════════════════════════════════════════════════════════
def plot_p4(df):
    pdf   = fy_shares(df)
    pivot = pdf.pivot(index="FY", columns="Mode", values="Share").fillna(0)
    tot   = pdf.groupby("Mode")["Volume"].sum().sort_values(ascending=False)
    top3  = tot.index[:3].tolist()
    col_order = top3 + [m for m in tot.index if m not in top3]
    pivot = pivot[[c for c in col_order if c in pivot.columns]]
    fy_labels = list(pivot.index)

    fig = plt.figure(figsize=(17, 7))
    gs  = gridspec.GridSpec(1, 2, wspace=0.32)
    ax1 = fig.add_subplot(gs[0])
    ax2 = fig.add_subplot(gs[1])

    fig.suptitle(
        "P4 — Pareto Analysis: Payment Mode Volume Share  ·  FY2020 – FY2025\n"
        "★ = Top-3 modes by cumulative transaction volume",
        fontsize=12, fontweight="bold", color=PALETTE["text"],
    )

    # ── Stacked bar ──
    x_pos   = np.arange(len(fy_labels))
    bottoms = np.zeros(len(fy_labels))
    for mode in col_order:
        if mode not in pivot.columns:
            continue
        vals   = pivot[mode].values
        is_top = mode in top3
        color  = PALETTE.get(mode, "#CCCCCC")
        ax1.bar(x_pos, vals, bottom=bottoms,
                color=color, alpha=0.88 if is_top else 0.30,
                edgecolor="white", linewidth=0.6, width=0.62,
                label=f"{'★ ' if is_top else ''}{mode}", zorder=3)
        for i, (v, b) in enumerate(zip(vals, bottoms)):
            if v > 4.0:
                ax1.text(i, b + v/2, f"{v:.1f}%",
                         ha="center", va="center",
                         fontsize=7.5 if is_top else 6.5,
                         fontweight="bold" if is_top else "normal",
                         color="white" if is_top else "#555555", zorder=4)
        bottoms += vals

    ax1.axhline(80, color="#333333", lw=1.2, ls="--", alpha=0.55, label="80% threshold")
    ax1.set_xticks(x_pos); ax1.set_xticklabels(fy_labels, fontsize=9)
    ax1.set_ylabel("Volume Share (%)", fontsize=9)
    ax1.set_ylim(0, 108)
    ax1.set_title("Volume Share by Fiscal Year  (stacked)", fontsize=10)
    ax1.legend(fontsize=7.5, loc="lower right", ncol=2, framealpha=0.95)
    ax1.grid(axis="y", ls=":", alpha=0.35)

    # ── FY2025 Pareto ──
    fy25 = pdf[pdf["FY"]=="FY25"].sort_values("Share", ascending=False)
    mods = fy25["Mode"].tolist()
    shrs = fy25["Share"].values
    cum  = np.cumsum(shrs)

    bar_col = [PALETTE.get(m,"#CCCCCC") for m in mods]
    ax2.bar(mods, shrs, color=bar_col, alpha=0.88, edgecolor="white", width=0.6, zorder=3)
    ax3 = ax2.twinx()
    ax3.plot(mods, cum, color="#222222", lw=2.2, marker="o", ms=7, zorder=5, label="Cumulative %")
    ax3.axhline(80, color="#333333", lw=1.2, ls="--", alpha=0.5)
    ax3.set_ylabel("Cumulative Volume Share (%)", fontsize=9)
    ax3.set_ylim(0, 112)
    ax3.legend(fontsize=8, loc="center right")

    for i, (m, s, c) in enumerate(zip(mods, shrs, cum)):
        ax2.text(i, s+0.8, f"{s:.1f}%", ha="center", va="bottom",
                 fontsize=8, fontweight="bold" if m in top3 else "normal",
                 color=PALETTE.get(m,"#333"))
        ax3.text(i, c+1.8, f"{c:.0f}%", ha="center", va="bottom",
                 fontsize=6.8, color="#444444")

    ax2.set_ylabel("Volume Share (%)", fontsize=9)
    ax2.set_title("FY2025 Pareto  (cumulative share →)", fontsize=10)
    ax2.grid(axis="y", ls=":", alpha=0.35)
    ax2.set_xticklabels(mods, fontsize=8.5)

    # Arrow annotation for top-3
    top3_cum = cum[2] if len(cum) > 2 else cum[-1]
    ax3.annotate(
        f"Top-3: {', '.join(top3[:3])}\n= {top3_cum:.1f}% of FY25 volume",
        xy=(2, top3_cum), xytext=(3.4, top3_cum-22),
        fontsize=8, color="#1A1A2E",
        arrowprops=dict(arrowstyle="->", color="#555555", lw=1.3),
        bbox=dict(boxstyle="round,pad=0.35", fc=PALETTE["anno"],
                  ec=PALETTE["anno_border"], alpha=0.95),
        zorder=8,
    )

    _source_note(fig)
    plt.tight_layout()
    savefig(fig, "P4_pareto_volume_share.png")

    # ── P4c trajectory ──
    fig2, ax4 = plt.subplots(figsize=(11, 5.5))
    fig2.suptitle(
        "P4c — Top-3 Mode Volume Share Trajectory  ·  FY2020 – FY2025",
        fontsize=12, fontweight="bold",
    )
    for mode in top3:
        v  = pivot[mode].values if mode in pivot.columns else np.zeros(len(fy_labels))
        c  = PALETTE.get(mode,"#888888")
        ax4.plot(fy_labels, v, marker="o", lw=2.6, color=c,
                 label=f"★ {mode}", ms=9, zorder=4)
        for i, val in enumerate(v):
            offset = 5 if val < 90 else -8
            ax4.annotate(f"{val:.1f}%", (i, val),
                         textcoords="offset points", xytext=(0, offset),
                         ha="center", fontsize=9, color=c, fontweight="bold")
    ax4.axhline(80, color="#333333", lw=1.1, ls="--", alpha=0.5, label="80% threshold")
    ax4.set_ylabel("Volume Share (%)", fontsize=9)
    ax4.set_ylim(0, 100)
    ax4.legend(fontsize=10, loc="upper left")
    ax4.grid(ls=":", alpha=0.35)
    _source_note(fig2)
    plt.tight_layout()
    savefig(fig2, "P4c_top3_trajectory.png")

# ═════════════════════════════════════════════════════════════════════════════
#  PSTAT — STATIONARITY
# ═════════════════════════════════════════════════════════════════════════════
def plot_stationarity(df):
    modes, orders, pvals = [], [], []
    for mode in ALL_MODES:
        s = df[df["Payment_Mode"]==mode].set_index("Date")["Volume"].dropna().sort_index()
        if len(s) < 15:
            continue
        log_s = np.log1p(s)
        _, d, p = _enforce_stationarity(log_s)
        modes.append(mode)
        orders.append(d)
        pvals.append(float(np.clip(p, 1e-6, 1.0)))

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5.5))
    fig.suptitle(
        "P_STAT — Stationarity Enforcement  ·  ADF Test on Seasonal-Differenced log(Volume)\n"
        "Seasonal diff (lag=12) applied first; first-diff added where needed",
        fontsize=11, fontweight="bold",
    )

    # ADF p-values
    bar_cols = [PALETTE.get(m,"#888") for m in modes]
    sig_cols = ["#2A9D8F" if p < ALPHA else "#D62728" for p in pvals]
    bars = ax1.barh(modes, pvals, color=sig_cols, edgecolor="white", height=0.55)
    ax1.axvline(ALPHA, color="#333", lw=1.5, ls="--", label=f"α = {ALPHA}")
    ax1.set_xlabel("ADF p-value  (post seasonal differencing)")
    ax1.set_title("ADF p-value by Mode", fontsize=10)
    ax1.legend(fontsize=9)
    for bar, p in zip(bars, pvals):
        status = "✓ stationary" if p < ALPHA else "✗ non-stationary"
        ax1.text(p + 0.003, bar.get_y() + bar.get_height()/2,
                 f"{p:.4f}  {status}", va="center", fontsize=8.5)
    ax1.set_xlim(0, max(pvals)*1.6+0.05)

    # Integration order
    ax2.bar(modes, orders,
            color=[PALETTE.get(m,"#888") for m in modes],
            edgecolor="white", width=0.5)
    ax2.set_ylabel("Additional 1st-diffs after seasonal diff", fontsize=8)
    ax2.set_title("Integration Order (extra diffs needed)", fontsize=10)
    ax2.set_yticks([0,1,2])
    ax2.set_yticklabels(["0 — stationary after\nseasonal diff",
                          "1 — one extra\n1st-diff",
                          "2 — two extra\n1st-diffs"], fontsize=8)
    ax2.grid(axis="y", ls=":", alpha=0.3)
    for i, (m, o) in enumerate(zip(modes, orders)):
        ax2.text(i, o+0.04, str(o), ha="center", fontsize=10, fontweight="bold")

    _source_note(fig)
    plt.tight_layout()
    savefig(fig, "P_stat_stationarity.png")

# ═════════════════════════════════════════════════════════════════════════════
#  PBREAK — STRUCTURAL BREAKS
# ═════════════════════════════════════════════════════════════════════════════
def _chow(y, break_idx):
    n, k = len(y), 2
    X = add_constant(np.arange(n).reshape(-1,1))
    rss_pool = np.sum(OLS(y,X).fit().resid**2)
    def rss_seg(ys, Xs):
        return np.nan if len(ys) < k+2 else np.sum(OLS(ys,Xs).fit().resid**2)
    r1 = rss_seg(y[:break_idx], X[:break_idx])
    r2 = rss_seg(y[break_idx:], X[break_idx:])
    if np.isnan(r1) or np.isnan(r2):
        return np.nan, np.nan
    df2 = n - 2*k
    if df2 <= 0:
        return np.nan, np.nan
    F = ((rss_pool - r1 - r2) / k) / ((r1+r2) / df2)
    p = 1 - scipy_stats.f.cdf(F, k, df2)
    return F, p

def plot_structural_breaks(df):
    cand_ts = [pd.Timestamp(d) for d in CHOW_CANDIDATES]

    stat_series = {}
    for mode in ALL_MODES:
        s = df[df["Payment_Mode"]==mode].set_index("Date")["Volume"].dropna().sort_index()
        if len(s) < 15:
            continue
        ts, _, _ = _enforce_stationarity(np.log1p(s))
        stat_series[mode] = ts.dropna()

    fig, axes = plt.subplots(2, 3, figsize=(17, 10))
    fig.suptitle(
        "P_BREAK — Structural Break Detection  ·  Chow Test + PELT\n"
        "Series: seasonally-differenced, stationarity-enforced log(Volume)\n"
        "Red dashed = Chow-significant (p<0.05)  ·  Green dotted = PELT breakpoint",
        fontsize=11, fontweight="bold",
    )

    for ax_i, mode in enumerate(list(stat_series.keys())[:6]):
        ax    = axes.flatten()[ax_i]
        s     = stat_series[mode]
        dates = s.index
        y     = s.values
        n     = len(y)
        color = PALETTE.get(mode,"#888")

        ax.plot(dates, y, color=color, lw=1.8, zorder=3)
        ax.fill_between(dates, y, alpha=0.12, color=color)

        chow_sig = []
        for cdt in cand_ts:
            if not (dates.min() < cdt < dates.max()):
                continue
            idx = int(np.searchsorted(dates, cdt))
            if idx < 4 or idx > n-4:
                continue
            F, p = _chow(y, idx)
            if not np.isnan(p) and p < ALPHA:
                chow_sig.append((cdt, F, p))
                ax.axvline(cdt, color="crimson", lw=1.8, ls="--", alpha=0.9, zorder=5)
                ax.text(cdt, y.max()*0.92,
                        f"F={F:.1f}\np={p:.3f}",
                        color="crimson", fontsize=6.5, rotation=90, va="top",
                        bbox=dict(boxstyle="round,pad=0.12",fc="white",alpha=0.75,ec="none"),
                        zorder=6)

        pelt_breaks = []
        if HAS_RUPTURES and n > 20:
            try:
                algo = rpt.Pelt(model="rbf", min_size=6, jump=1).fit(y)
                idxs = [b for b in algo.predict(pen=3.0) if b < n]
                pelt_breaks = [dates[b-1] for b in idxs]
                for bt in pelt_breaks:
                    ax.axvline(bt, color="#1A8A4A", lw=1.3, ls=":", alpha=0.85, zorder=4)
            except Exception:
                pass

        ax.set_title(mode, fontsize=11, fontweight="bold", color=color)
        ax.xaxis.set_major_formatter(mdates.DateFormatter("%b-%y"))
        ax.xaxis.set_major_locator(mdates.MonthLocator(bymonth=[1,7]))
        plt.setp(ax.xaxis.get_majorticklabels(), rotation=35, ha="right", fontsize=6.5)
        ax.set_ylabel("Stationary log(Volume)", fontsize=7.5)
        ax.grid(axis="y", ls=":", alpha=0.3)

        summary = (
            f"Chow breaks: {len(chow_sig)}  "
            + (("→ " + ", ".join(c[0].strftime("%b-%Y") for c in chow_sig)) if chow_sig else "→ none")
            + f"\nPELT detections: {len(pelt_breaks)}"
        )
        _insight_box(ax, summary, x=0.03, y=0.03, fontsize=6.8)

        legend_h = [
            Line2D([0],[0], color=color,    lw=2.0,   label=f"{mode} series"),
            Line2D([0],[0], color="crimson",lw=1.8,ls="--", label="Chow sig."),
            Line2D([0],[0], color="#1A8A4A",lw=1.3,ls=":",  label="PELT break"),
        ]
        ax.legend(handles=legend_h, fontsize=6.5, loc="upper right")

    _source_note(fig)
    plt.tight_layout(rect=[0,0,1,0.94])
    savefig(fig, "P_break_structural.png")

# ═════════════════════════════════════════════════════════════════════════════
#  PGRANGER — GRANGER CAUSALITY  (heatmap + bar grid)
# ═════════════════════════════════════════════════════════════════════════════
def plot_granger(df):
    MAX_LAG = 4

    stat = {}
    for mode in ["UPI","Debit","NEFT","IMPS"]:
        s = df[df["Payment_Mode"]==mode].set_index("Date")["Volume"].dropna().sort_index()
        if len(s) < 20:
            continue
        ts, order, _ = _enforce_stationarity(np.log1p(s))
        stat[mode] = (ts.dropna(), order)

    if "UPI" not in stat:
        print("  ⚠  UPI not available for Granger — skipping")
        return

    upi_s, upi_d = stat["UPI"]

    targets = {
        "Debit  (⚠ proxy)": ("Debit", PALETTE["Debit"]),
        "NEFT  (⚠ proxy)":  ("NEFT",  PALETTE["NEFT"]),
        "IMPS":              ("IMPS",  PALETTE["IMPS"]),
    }
    segments = {
        "Full sample":   (None, None),
        "Pre-Apr 2023":  (None, BREAK_DATE),
        "Post-Apr 2023": (BREAK_DATE, None),
    }

    n_tgt = len(targets)
    n_seg = len(segments)

    fig = plt.figure(figsize=(17, 12))
    outer_gs = gridspec.GridSpec(n_tgt+1, 1, height_ratios=[0.05]+[1]*n_tgt,
                                  hspace=0.55)
    fig.suptitle(
        "P_GRANGER — Granger Predictability: UPI Volume → Debit / NEFT / IMPS\n"
        "Null: UPI lags add no predictive value beyond target's own lags\n"
        "Filled bar = F > F-crit (p<0.05)  ·  This is predictive directionality ONLY, NOT causation",
        fontsize=11, fontweight="bold",
    )

    lags_list = list(range(1, MAX_LAG+1))

    for row_i, (tgt_label, (tgt_mode, tgt_color)) in enumerate(targets.items()):
        inner_gs = gridspec.GridSpecFromSubplotSpec(1, n_seg,
                        subplot_spec=outer_gs[row_i+1], wspace=0.35)

        if tgt_mode not in stat:
            continue

        tgt_s, tgt_d = stat[tgt_mode]
        upi_aligned = upi_s.copy()
        tgt_aligned = tgt_s.copy()

        # Align integration orders
        diff_extra = abs(upi_d - tgt_d)
        if upi_d < tgt_d:
            for _ in range(diff_extra):
                upi_aligned = upi_aligned.diff().dropna()
        elif tgt_d < upi_d:
            for _ in range(diff_extra):
                tgt_aligned = tgt_aligned.diff().dropna()

        for col_j, (seg_label, (seg_start, seg_end)) in enumerate(segments.items()):
            ax = fig.add_subplot(inner_gs[col_j])

            u = upi_aligned.copy()
            t = tgt_aligned.copy()
            if seg_start: u = u[u.index >= seg_start]; t = t[t.index >= seg_start]
            if seg_end:   u = u[u.index <  seg_end];   t = t[t.index <  seg_end]

            combined = pd.concat([u.rename("UPI"), t.rename("TGT")], axis=1).dropna()
            n_obs    = len(combined)

            ax.set_title(f"{seg_label}  (n={n_obs})", fontsize=8, fontweight="bold")

            if n_obs < MAX_LAG + 8:
                ax.text(0.5, 0.5, f"Insufficient data\nn={n_obs}",
                        ha="center", va="center",
                        transform=ax.transAxes, fontsize=9, color="#999999",
                        bbox=dict(boxstyle="round", fc="#f5f5f5", ec="#cccccc"))
                ax.set_xticks([]); ax.set_yticks([])
                continue

            try:
                gc = grangercausalitytests(combined[["TGT","UPI"]].values,
                                           maxlag=MAX_LAG, verbose=False)
            except Exception as e:
                ax.text(0.5, 0.5, f"Test failed\n{str(e)[:40]}",
                        ha="center", va="center", transform=ax.transAxes, fontsize=8)
                continue

            fstats = [gc[lag][0]["ssr_ftest"][0] for lag in lags_list]
            pvals  = [gc[lag][0]["ssr_ftest"][1] for lag in lags_list]

            df_num = 1
            df_den = max(n_obs - MAX_LAG - 2, 1)
            f_crit = scipy_stats.f.ppf(1-ALPHA, df_num, df_den)

            bar_alpha = [0.85 if p < ALPHA else 0.22 for p in pvals]
            for lag, F, p, alp in zip(lags_list, fstats, pvals, bar_alpha):
                ax.bar(lag, F, color=tgt_color, alpha=alp,
                       edgecolor=tgt_color, linewidth=0.7, width=0.6, zorder=3)
                yoff = F * 0.06 + f_crit * 0.08
                ax.text(lag, F + yoff,
                        f"p={p:.2f}\n{sig_stars(p)}",
                        ha="center", va="bottom",
                        fontsize=6.5,
                        color="#111111" if p < ALPHA else "#aaaaaa",
                        fontweight="bold" if p < ALPHA else "normal",
                        linespacing=1.2)

            ax.axhline(f_crit, color="crimson", lw=1.4, ls="--",
                       label=f"F-crit={f_crit:.2f}", zorder=5)
            ax.set_xlim(0.4, MAX_LAG+0.6)
            ax.set_xticks(lags_list)
            ax.set_xlabel("Lag (months)", fontsize=7.5)
            ax.set_ylabel("F-statistic", fontsize=7.5)
            ax.legend(fontsize=7, loc="upper right")
            ax.grid(axis="y", ls=":", alpha=0.3)

            # Dynamic ylim
            all_f = fstats + [f_crit]
            ax.set_ylim(0, max(all_f)*1.55 + 2)

        # Row label
        fig.text(0.005, 0.84 - row_i*0.285,
                 f"→ {tgt_label}", fontsize=9, fontweight="bold",
                 color=tgt_color, va="center", rotation=90)

    fig.text(0.01, 0.005,
             "⚠ Debit and NEFT include behaviours beyond ATM/PPI — absence of signal does not confirm no effect\n"
             "⚠ Post-Apr-2023 segment has <25 months — treat results as exploratory",
             fontsize=7, color="#666666")
    _source_note(fig)
    plt.tight_layout(rect=[0.025, 0.02, 1, 0.93])
    savefig(fig, "P_granger_causality.png")

# ═════════════════════════════════════════════════════════════════════════════
#  MAIN
# ═════════════════════════════════════════════════════════════════════════════
def main():
    csv_path = Path(INPUT_CSV)
    if not csv_path.exists():
        print(f"\n  ✗  '{INPUT_CSV}' not found in current directory.")
        print("     Place rbi_payments_clean.csv here and re-run.\n")
        sys.exit(1)

    print(f"\n  Loading  : {INPUT_CSV}")
    df = pd.read_csv(INPUT_CSV, parse_dates=["Date"])

    # Ensure log_volume exists (derived column)
    if "log_volume" not in df.columns:
        df["log_volume"] = np.log1p(df["Volume"])

    print(f"  Shape    : {df.shape}")
    print(f"  Range    : {df['Date'].min().strftime('%b %Y')} → {df['Date'].max().strftime('%b %Y')}")
    print(f"  Modes    : {sorted(df['Payment_Mode'].unique())}")
    print(f"  Output   : {OUT_DIR.resolve()}/\n")

    print("  Generating Phase 3 diagnostic plots …")
    plot_stationarity(df)
    plot_structural_breaks(df)
    plot_granger(df)
    plot_p1(df)
    plot_p2(df)
    plot_p3(df)
    plot_p4(df)

    n = len(list(OUT_DIR.glob("*.png")))
    print(f"\n  ✓  Done — {n} PNGs saved to  {OUT_DIR}/\n")

    print("  Plot inventory:")
    for p in sorted(OUT_DIR.glob("*.png")):
        print(f"      {p.name}")

if __name__ == "__main__":
    main()


  Loading  : rbi_payments_clean.csv
  Shape    : (390, 14)
  Range    : Nov 2019 → Mar 2025
  Modes    : ['Credit', 'Debit', 'IMPS', 'NEFT', 'RTGS', 'UPI']
  Output   : /content/plots_phase3/

  Generating Phase 3 diagnostic plots …
  ✓  plots_phase3/P_stat_stationarity.png
  ✓  plots_phase3/P_break_structural.png
  ✓  plots_phase3/P_granger_causality.png
  ✓  plots_phase3/P1_event_overlay_timeline.png
  ✓  plots_phase3/P2_covid_impact.png
  ✓  plots_phase3/P3_interchange_impact.png
  ✓  plots_phase3/P4_pareto_volume_share.png
  ✓  plots_phase3/P4c_top3_trajectory.png

  ✓  Done — 8 PNGs saved to  plots_phase3/

  Plot inventory:
      P1_event_overlay_timeline.png
      P2_covid_impact.png
      P3_interchange_impact.png
      P4_pareto_volume_share.png
      P4c_top3_trajectory.png
      P_break_structural.png
      P_granger_causality.png
      P_stat_stationarity.png
